## Filter the Blast hits for >50% coverage and >30% identity. 

In [2]:
from Bio import SeqIO
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 36.7 MB/s eta 0:00:00


In [3]:
# import the fasta file for your target protein
target_fasta = list(SeqIO.parse('PE6c_RT.fasta','fasta'))
name = target_fasta[0].id

In [3]:
# import the data table output (.tsv) of the initial blast search
df = pd.read_csv('PE6c_RT_blast_out_table.tsv',sep='\t',header=None)
df.columns = ['query','id','match','length','mismatch','gapopen','qstart','qend','sstart','send','Eval','bitscore']

# filter the blast output for 50% coverage and 30% identity 
filtered_ids = []
prot_length = len(target_fasta[0].seq)

for index, row in df.iterrows():
    if row['match'] > 30:
        coverage = (prot_length)/2
        if row['length'] > coverage:
            filtered_ids.append(row['id'])

In [5]:
# analyz the filtered sequences by importing the fasta file of Blast hit sequences
blast_fasta = list(SeqIO.parse('PE6c_RT_hits.fasta','fasta'))

filtered_seqs = {}
entries = []
entries.append(target_fasta[0]) #add target protein to the top

for record in blast_fasta:
    if record.id in filtered_ids:
        filtered_seqs[record.id] = record.seq
        entries.append(record)
        
SeqIO.write(entries, "filtered_hits.fasta", "fasta")

519

## Make MSA from the fasta file

The next steps will make a multiple sequence aligment with clustal omega. If you don't have clustal omega installed you can make the multiple alignment of the filtered_hits.fasta using Geneious, move the file to this directory, then skip ahead to the next note

count the residues at each position of the BoNT/F alignment. Write a function to generate a list of residues with conservation above a specified cutoff.

In [4]:
from Bio.Align.Applications import ClustalOmegaCommandline
from Bio import AlignIO

/opt/anaconda3/envs/ml/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [5]:
'''multiple sequence alignment using Clustal Omega'''
clustalomega_cline = ClustalOmegaCommandline(infile='PE6c_RT_filtered_hits.fasta', outfile='PE6c_RT_filtered_aligned.fasta',force=True)
clustalomega_cline()

('', '')

In [6]:
'''count the residues at each position in the alignment'''
# Parse the resulting alignment file
alignment = AlignIO.read('PE6c_RT_filtered_aligned.fasta', "fasta")

# Add up the counts of each non-gap AA at each position 
positions_dict = {}
for pos in range(alignment.get_alignment_length()):
    dict = {}
    for i in range(len(alignment)):
        if alignment[i][pos] != '-':
            if alignment[i][pos] not in dict:
                dict[alignment[i][pos]] = 1
            else:
                dict[alignment[i][pos]] += 1
    
    positions_dict[pos] = dict
    
# Determine the fraction consensus and most abundant AA (not including gaps in count)
consensus = []
consensus_AA = []
for i in range(len(positions_dict)):
    key_with_highest_value = max(positions_dict[i], key=lambda key: positions_dict[i][key])
    consensus_AA.append(key_with_highest_value)
    
    total = 0
    for value in positions_dict[i].values():
        total += value
    consensus.append(max(positions_dict[i].values())/total)

## Make the constraint list 


In [10]:
'''import the distance residues text file generated from AT_Tutorial_1.ipnyb'''
# Step 1: Define the file path
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_fixed_residues_20A.txt'

# Step 2: Read the contents of the file
with open(file_path, 'r') as infile:
    data = infile.read()

# Step 3: Convert the string data to a list, splitting by tabs
PDB_fixed_res = data.split('\n')

# Step 4: Convert residue indices to integers
PDB_fixed_res = [int(res.strip()) for res in PDB_fixed_res if res.strip().isdigit()]
man_fixed_res = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18] # fixed the N terminal of the PE6c becuase it is too disordered in the AF structure

In [11]:
'''Define the function to choose residues to fix based on selected conservation cutoffs and combine with the distance constraint list'''
def get_res(cutoffList,distance_cutoff):
    output_dict = {}

    #choose conserved residues to fix based on cutoff
    for cutoff in cutoffList:
        fixed_cons = []
        x = 0
        sequence = str(alignment[0].seq)
        for i,res in enumerate(sequence):
            if res != '-':
                x += 1
                if res == consensus_AA[i]:
                    if consensus[i] >= cutoff:
                        fixed_cons.append(x)
    
        frac = len(fixed_cons)/len(target_fasta[0].seq)
    
        #make list of the fixed residues from distance or conservation requirement
        all_fixed_residues = []
        for x in fixed_cons:
            all_fixed_residues.append(x)
        for y in PDB_fixed_res:
            if y not in all_fixed_residues:
                all_fixed_residues.append(y)  
        for z in man_fixed_res:
            if z not in all_fixed_residues:
                all_fixed_residues.append(z)
                
        all_fixed_residues.sort()
        
        frac_all_fixed = len(all_fixed_residues)/len(target_fasta[0].seq)

        all_fixed_res_str = ''
        for i in all_fixed_residues:
            all_fixed_res_str= all_fixed_res_str + str(i) + ' '
    
        '''pymol commands to select residues'''
        # select residues to fix by conservation
        pymol_command_conserve = 'select conserve, chain A and resi '
        for x in fixed_cons:
          pymol_command_conserve = pymol_command_conserve + str(x) + '+'

        # select residues to fix by distance
        pymol_command_bind = 'select bind, chain A and resi '
        for x in PDB_fixed_res:
          pymol_command_bind = pymol_command_bind + str(x) + '+'

        # select residues to fix manually
        #pymol_command_man = 'select manual, chain A and resi '
        #for x in man_fixed_res:
          #pymol_command_man = pymol_command_man + str(x) + '+'

        output_dict[cutoff] = [all_fixed_res_str,all_fixed_residues, fixed_cons,frac, frac_all_fixed, pymol_command_conserve, pymol_command_bind]

    df = pd.DataFrame.from_dict(output_dict)
    cols = ['MPNN fix string','all fixed','fixed by conservation','fraction fixed by conservation','fraction all res fixed','pymol conserved','pymol bind']
    df=df.T
    df.columns=cols
    df.to_csv(name+'_'+str(distance_cutoff)+'_constraints.csv')

In [12]:
'''Run the function with your desired inputs. 
The first argument is a list of your conservation cutoffs (30% and 60% in the example) 
and the second argument is the distance cutoff you used in your distance analysis'''

get_res([.25, 0.5],20)

A csv file should have been generated with the list of fixed residues for each conservation cutoff and the particular distance cutoff. To generate constraints for different distance cutoffs, rerun this script with the distance-constrained residue lists you generated for those cutoffs.

In [2]:
import prody


ImportError: cannot import name 'tarfile' from 'backports' (/opt/anaconda3/envs/ml/lib/python3.10/site-packages/backports/__init__.py)